# Assignment 9 – Local Binary Pattern and Image Denoising

**Assignment tasks**
1. Implement Local Binary Pattern (LBP) on a grayscale image.
2. Denoise images corrupted by Gaussian and Speckle noise using:
   - Wiener filter
   - Median filter
   - Gaussian filter
   - Bilateral filter

This notebook applies 3×3 and 5×5 windows, displays results, and compares outputs using MSE and PSNR.

**Input image:** Put `input_image.jpg` in the same folder as this notebook. If it is not present, the notebook automatically uses the sample `camera` image from scikit-image.

## 1. Import libraries

In [ ]:
# If any library is missing, uncomment and run:
# %pip install numpy matplotlib opencv-python scipy scikit-image

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import wiener
from skimage import data
from skimage.feature import local_binary_pattern
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio

print("Libraries imported successfully.")

## 2. Load image and convert to grayscale

In [ ]:
IMAGE_PATH = "input_image.jpg"  # Change this to your image filename

if os.path.exists(IMAGE_PATH):
    image = cv2.imread(IMAGE_PATH)
    if image is None:
        raise ValueError("Could not read the image. Check IMAGE_PATH.")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    print("Loaded:", IMAGE_PATH)
else:
    gray = np.asarray(data.camera(), dtype=np.uint8)
    print("input_image.jpg not found; using the built-in sample camera image.")

gray = np.asarray(gray, dtype=np.uint8)
gray_float = gray.astype(np.float32) / 255.0

plt.figure(figsize=(7, 5))
plt.imshow(gray, cmap="gray")
plt.title("Original Grayscale Image")
plt.axis("off")
plt.show()

print("Image shape:", gray.shape)
print("Data type:", gray.dtype)

## 3. Local Binary Pattern (LBP)

LBP compares each neighbor's intensity with the center pixel. A neighbor is assigned 1 if its intensity is greater than or equal to the center, otherwise 0. The resulting binary pattern is converted into a numerical label.

Here, 8 sampling points and radius 1 are used with the `uniform` LBP method.

In [ ]:
radius = 1
n_points = 8 * radius

lbp_image = local_binary_pattern(gray, n_points, radius, method="uniform")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original Grayscale Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(lbp_image, cmap="gray")
plt.title("Local Binary Pattern")
plt.axis("off")
plt.tight_layout()
plt.show()

print("LBP completed.")
print("LBP output shape:", lbp_image.shape)
print("Unique LBP labels:", len(np.unique(lbp_image)))

## 4. Add Gaussian noise and Speckle noise

- **Gaussian noise:** additive noise sampled from a Gaussian distribution.
- **Speckle noise:** multiplicative noise; the random noise is multiplied by the image intensity.

The fixed random seed makes the results reproducible.

In [ ]:
rng = np.random.default_rng(42)
NOISE_STD = 0.10

# Gaussian (additive) noise
gaussian_noise = rng.normal(0.0, NOISE_STD, gray_float.shape)
noisy_gaussian = np.clip(gray_float + gaussian_noise, 0.0, 1.0)

# Speckle (multiplicative) noise
speckle_noise = rng.normal(0.0, NOISE_STD, gray_float.shape)
noisy_speckle = np.clip(gray_float + gray_float * speckle_noise, 0.0, 1.0)

def to_uint8(img):
    return np.clip(img * 255.0, 0, 255).astype(np.uint8)

plt.figure(figsize=(14, 5))
for i, (title, img) in enumerate([
    ("Original", gray_float),
    ("Gaussian Noise", noisy_gaussian),
    ("Speckle Noise", noisy_speckle)
], 1):
    plt.subplot(1, 3, i)
    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Define the four denoising filters

1. **Wiener filter:** uses local mean and variance to estimate and reduce noise.
2. **Median filter:** replaces each pixel with the median value in its neighborhood.
3. **Gaussian filter:** smooths the image using a Gaussian-weighted kernel.
4. **Bilateral filter:** considers spatial distance and intensity similarity, helping preserve edges.

In [ ]:
def apply_wiener(img, ksize):
    result = wiener(img.astype(np.float32), mysize=(ksize, ksize))
    return np.clip(result, 0.0, 1.0)

def apply_median(img, ksize):
    result = cv2.medianBlur(to_uint8(img), ksize)
    return result.astype(np.float32) / 255.0

def apply_gaussian(img, ksize):
    result = cv2.GaussianBlur(to_uint8(img), (ksize, ksize), sigmaX=0)
    return result.astype(np.float32) / 255.0

def apply_bilateral(img, ksize):
    result = cv2.bilateralFilter(
        to_uint8(img), d=ksize, sigmaColor=50, sigmaSpace=50
    )
    return result.astype(np.float32) / 255.0

filters = {
    "Wiener": apply_wiener,
    "Median": apply_median,
    "Gaussian": apply_gaussian,
    "Bilateral": apply_bilateral
}
print("All four filters are defined.")

## 6. Apply filters to the Gaussian-noisy image

In [ ]:
gaussian_results = {}
for filter_name, filter_function in filters.items():
    for window in (3, 5):
        gaussian_results[(filter_name, window)] = filter_function(noisy_gaussian, window)

fig, axes = plt.subplots(3, 3, figsize=(13, 12))
display_items = [
    ("Original", gray_float),
    ("Gaussian noisy", noisy_gaussian),
    ("Wiener 3x3", gaussian_results[("Wiener", 3)]),
    ("Median 3x3", gaussian_results[("Median", 3)]),
    ("Gaussian 3x3", gaussian_results[("Gaussian", 3)]),
    ("Bilateral 3x3", gaussian_results[("Bilateral", 3)]),
    ("Wiener 5x5", gaussian_results[("Wiener", 5)]),
    ("Median 5x5", gaussian_results[("Median", 5)]),
    ("Gaussian 5x5", gaussian_results[("Gaussian", 5)])
]
for ax, (title, img) in zip(axes.flat, display_items):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
for i, name in enumerate(["Gaussian", "Bilateral"], 1):
    plt.subplot(1, 2, i)
    plt.imshow(gaussian_results[(name, 5)], cmap="gray", vmin=0, vmax=1)
    plt.title(f"{name} filter 5x5")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Apply filters to the Speckle-noisy image

In [ ]:
speckle_results = {}
for filter_name, filter_function in filters.items():
    for window in (3, 5):
        speckle_results[(filter_name, window)] = filter_function(noisy_speckle, window)

fig, axes = plt.subplots(3, 3, figsize=(13, 12))
display_items = [
    ("Original", gray_float),
    ("Speckle noisy", noisy_speckle),
    ("Wiener 3x3", speckle_results[("Wiener", 3)]),
    ("Median 3x3", speckle_results[("Median", 3)]),
    ("Gaussian 3x3", speckle_results[("Gaussian", 3)]),
    ("Bilateral 3x3", speckle_results[("Bilateral", 3)]),
    ("Wiener 5x5", speckle_results[("Wiener", 5)]),
    ("Median 5x5", speckle_results[("Median", 5)]),
    ("Gaussian 5x5", speckle_results[("Gaussian", 5)])
]
for ax, (title, img) in zip(axes.flat, display_items):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
for i, name in enumerate(["Gaussian", "Bilateral"], 1):
    plt.subplot(1, 2, i)
    plt.imshow(speckle_results[(name, 5)], cmap="gray", vmin=0, vmax=1)
    plt.title(f"{name} filter 5x5")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Compare filter performance using MSE and PSNR

- **MSE (Mean Squared Error):** average squared difference between the original and filtered image. Lower is better.
- **PSNR (Peak Signal-to-Noise Ratio):** reconstruction quality in decibels. Higher is generally better.

The comparison uses the original clean image as the reference.

In [ ]:
def calculate_metrics(reference, result):
    mse_value = mean_squared_error(reference, result)
    psnr_value = peak_signal_noise_ratio(reference, result, data_range=1.0)
    return mse_value, psnr_value

print(f"{'Noise':<12} {'Filter':<12} {'Window':<8} {'MSE':>12} {'PSNR (dB)':>12}")
print("-" * 62)

for noise_name, result_set in [
    ("Gaussian", gaussian_results),
    ("Speckle", speckle_results)
]:
    for (filter_name, window), result in result_set.items():
        mse_value, psnr_value = calculate_metrics(gray_float, result)
        print(f"{noise_name:<12} {filter_name:<12} {window}x{window:<5} "
              f"{mse_value:>12.6f} {psnr_value:>12.3f}")

## 9. Observations and conclusion

**Observations**
1. LBP represents local texture by encoding comparisons between neighboring pixels and the center pixel.
2. Gaussian noise is additive, whereas speckle noise is multiplicative.
3. Wiener filtering uses local statistical information.
4. Median filtering uses the median of the local neighborhood.
5. Gaussian filtering smooths the image; excessive smoothing can blur details.
6. Bilateral filtering considers both spatial distance and intensity similarity, helping preserve edges.
7. A 5×5 window considers more pixels than a 3×3 window and may smooth more strongly, but can remove fine details.

**Conclusion**

In this assignment, Local Binary Pattern was implemented on a grayscale image. Gaussian and speckle noise were added, and Wiener, median, Gaussian, and bilateral filters were applied using 3×3 and 5×5 windows. The outputs were displayed and compared using MSE and PSNR. The best-performing filter depends on the input image and noise level, so the metric values from the run should be used when drawing a conclusion.

## 10. Viva questions and answers

**1. What is LBP?**  
Local Binary Pattern is a texture descriptor that compares neighboring pixel intensities with the center pixel and encodes the comparisons as a binary pattern.

**2. Why is grayscale used?**  
A grayscale image has one intensity value per pixel, making neighborhood comparisons straightforward.

**3. What is Gaussian noise?**  
Additive noise commonly modeled using a Gaussian (normal) probability distribution.

**4. What is speckle noise?**  
Multiplicative noise whose magnitude depends on the image intensity.

**5. What is a Wiener filter?**  
A statistical filter that uses local mean and variance to estimate and reduce noise.

**6. What does a median filter do?**  
It replaces each pixel with the median of the values in its local neighborhood.

**7. What is the purpose of a Gaussian filter?**  
It smooths an image using a Gaussian-weighted kernel.

**8. How is bilateral filtering different from Gaussian filtering?**  
Bilateral filtering considers intensity similarity as well as spatial distance, so it can preserve edges better.

**9. What happens when the window size increases from 3×3 to 5×5?**  
More neighbors are considered, often producing stronger smoothing but potentially removing fine details.

**10. What do MSE and PSNR indicate?**  
MSE measures average squared error (lower is better). PSNR measures reconstruction quality in decibels (higher is generally better).